In [24]:
# import cv2
# os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
# plt.rcParams['image.cmap'] = 'viridis'


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import tempfile
from openai import OpenAI
import base64
from PIL import Image
import io
import os 
from typing import List, Tuple, Dict
import torch
import cv2
from ultralytics import YOLO
from matplotlib import colors
from matplotlib import patches
from FoodDetection.config import PREDICT_ARGS
from FoodDetection.food_detection import load_and_resize_image, detect_objects, plot_results
from FoodSegmentation.segmentation import LoadSAMPredictor,run_sam_with_multiple_points
from MenuSearch.search_index import FoodImageDataset,load_labels,load_faiss_index,clip_transform
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
yolo_path = r'C:\Users\user\Documents\code\korean_food_detection\tools\best.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# index_path = r"C:\Users\user\Documents\GitHub\Korean_Food_Detection\tools\faiss_index_cheat.index"
# labels_path = r"C:\Users\user\Documents\GitHub\Korean_Food_Detection\tools\labels_cheat.npy"
sam_checkpoint = r"C:\Users\user\Documents\code\korean_food_detection\tools\sam_vit_b_01ec64.pth"

model_type = "vit_b"
yolo_model = YOLO(yolo_path)
sam_predictor = LoadSAMPredictor(sam_checkpoint, model_type, device='cuda', return_sam=False)
api_key=""

In [26]:
import asyncio
class OpenAIClassifier:
    def __init__(self, api_key: str):
        self.client = OpenAI(api_key=api_key)
    
    # def encode_image_array(self, image_array: np.ndarray) -> str:
    #     """Convert numpy array to base64 string"""
    #     image = Image.fromarray(image_array)
    #     buffered = io.BytesIO()
    #     image.save(buffered, format="JPEG")
    #     return base64.b64encode(buffered.getvalue()).decode('utf-8')
    def encode_image_array(self, image_array: np.ndarray, target_size: Tuple[int, int] = (128, 128)) -> str:
        """Convert numpy array to base64 string after resizing."""
        # Resize image with LANCZOS filter for high-quality downsampling
        image = Image.fromarray(image_array).resize(target_size, Image.LANCZOS)
        buffered = io.BytesIO()
        image.save(buffered, format="JPEG", quality=85)  # Adjust quality if needed
        return base64.b64encode(buffered.getvalue()).decode('utf-8')

    
    async def classify_image(self, image_array: np.ndarray) -> str:
        """Classify image using OpenAI Vision API"""
        try:
            base64_image = self.encode_image_array(image_array)
            
            response = self.client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": "What korean food item is shown in this image? Respond with just the food name (english name if possible) in a single word."
                            },
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{base64_image}"
                                }
                            }
                        ]
                    }
                ],
                max_tokens=50
            )
            
            return response.choices[0].message.content.strip().lower()
            
        except Exception as e:
            print(f"Classification error: {str(e)}")
            return "unknown"

# async def process_detections(
#     detections: np.ndarray,
#     confidences: np.ndarray,
#     image_np: np.ndarray,
#     sam_predictor,
#     classifier: OpenAIClassifier,
#     device: str
# ) -> Tuple[List[Dict], List[np.ndarray]]:
#     results = []
#     all_masks = []
    
#     for i, (bbox, conf) in enumerate(zip(detections, confidences)):
#         x1, y1, x2, y2 = map(int, bbox)
        
#         # Get crop of the detection
#         crop = image_np[y1:y2, x1:x2]
        
#         # Classify the crop using OpenAI Vision API
#         class_name = await classifier.classify_image(crop)
        
#         # Generate mask using SAM
#         input_box = np.array([x1, y1, x2, y2])
#         masks, _, _ = sam_predictor.predict(
#             point_coords=None,
#             point_labels=None,
#             box=input_box[None, :],
#             multimask_output=False,
#         )
        
#         results.append({
#             'bbox': [x1, y1, x2, y2],
#             'confidence': float(conf),
#             'class': class_name
#         })
#         all_masks.append(masks[0])
    
#     return results, all_masks


async def process_detections(
    detections: np.ndarray,
    confidences: np.ndarray,
    image_np: np.ndarray,
    sam_predictor,
    classifier: OpenAIClassifier,
    device: str
) -> Tuple[List[Dict], List[np.ndarray]]:
    results = []
    all_masks = []
    
    # List of classification tasks
    classification_tasks = []

    for i, (bbox, conf) in enumerate(zip(detections, confidences)):
        x1, y1, x2, y2 = map(int, bbox)
        
        # Get crop of the detection
        crop = image_np[y1:y2, x1:x2]
        
        # Create a classification task and add to the list
        classification_tasks.append(classifier.classify_image(crop))
        
        # Generate mask using SAM (synchronously)
        input_box = np.array([x1, y1, x2, y2])
        masks, _, _ = sam_predictor.predict(
            point_coords=None,
            point_labels=None,
            box=input_box[None, :],
            multimask_output=False,
        )
        
        all_masks.append(masks[0])

    # Run all classification tasks concurrently
    class_names = await asyncio.gather(*classification_tasks)
    
    # Combine results
    for i, (bbox, conf, class_name) in enumerate(zip(detections, confidences, class_names)):
        x1, y1, x2, y2 = map(int, bbox)
        results.append({
            'bbox': [x1, y1, x2, y2],
            'confidence': float(conf),
            'class': class_name
        })
    
    return results, all_masks

async def main(image_path: str, PREDICT_ARGS: dict, device: str):
    """Main function for food detection and classification"""
    # Initialize OpenAI classifier
    classifier = OpenAIClassifier(api_key=api_key)
    
    # Load and preprocess image
    image, image_np = load_and_resize_image(image_path)


    # Detect objects
    detection_results = detect_objects(yolo_model, image, PREDICT_ARGS)

    if len(detection_results) > 0:
        detections = detection_results[0].boxes.xyxy
        confidences = detection_results[0].boxes.conf

        # Generate random colors
        colors_list = list(colors.CSS4_COLORS.keys())
        np.random.seed(42)
        random_colors = np.random.choice(colors_list, size=len(detections), replace=False)

        # Set the image for SAM predictor
        sam_predictor.set_image(image_np)

        # Process detections with OpenAI classifier
        results, all_masks = await process_detections(
            detections,
            confidences,
            image_np,
            sam_predictor,
            classifier,
            device
        )

        # Plot results
        fig, ax = plt.subplots(1, figsize=(12, 8))
        plot_results(ax, image_np, results, all_masks, random_colors)

        # Save the image with detections
        result_image_path = tempfile.mktemp(suffix=".jpg")
        plt.savefig(result_image_path)
        plt.close()

        return result_image_path
    else:
        return None

def plot_results(ax, image_np, results, all_masks, random_colors):
    """Plot detection results with refined masks, labels, and bounding boxes."""
    ax.imshow(image_np)

    for i, result in enumerate(results):
        x1, y1, x2, y2 = result['bbox']
        class_name = result['class']
        confidence = result['confidence']
        
        # Display label with confidence
        label_with_confidence = f"{class_name or 'Unknown'} ({confidence:.2f})"
        ax.text(x1, y1 - 10, label_with_confidence, color='white', fontsize=12, 
                backgroundcolor=random_colors[i])

        # Draw bounding box
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=random_colors[i], facecolor='none'
        )
        ax.add_patch(rect)

        # Process and refine mask for better display
        mask = all_masks[i]
        binary_mask = (mask > 0.5).astype(np.uint8)

        # Morphological operations to clean up mask edges
        kernel = np.ones((5, 5), np.uint8)
        binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)
        binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel)

        # Find the largest contour for a cleaner mask display
        contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            largest_contour = max(contours, key=cv2.contourArea)
            refined_mask = np.zeros_like(binary_mask)
            cv2.drawContours(refined_mask, [largest_contour], 0, 1, -1)

            # Overlay mask with color and transparency
            color_rgba = colors.to_rgba(random_colors[i], alpha=0.4)
            mask_overlay = np.zeros((*refined_mask.shape, 4), dtype=np.float32)
            mask_overlay[refined_mask == 1] = color_rgba
            ax.imshow(mask_overlay)

    ax.axis('off')

async def run_food_detection_and_classification(image_path, PREDICT_ARGS, device):
    result_image_path = await main(image_path, PREDICT_ARGS, device)
    if result_image_path:
        return Image.open(result_image_path)
    else:
        return None

# Example usage


In [27]:
import asyncio
from typing import List, Dict, Tuple
import numpy as np
from PIL import Image
import io
import base64
import tempfile
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import colors
import cv2

class OpenAIClassifier:
    def __init__(self, api_key: str):
        self.client = OpenAI(api_key=api_key)
    
    def encode_image_array(self, image_array: np.ndarray, target_size: Tuple[int, int] = (128, 128)) -> str:
        """Convert numpy array to base64 string after resizing."""
        image = Image.fromarray(image_array).resize(target_size, Image.LANCZOS)
        buffered = io.BytesIO()
        image.save(buffered, format="JPEG", quality=85)
        return base64.b64encode(buffered.getvalue()).decode('utf-8')

    async def classify_image(self, image_array: np.ndarray, semaphore: asyncio.Semaphore) -> str:
        """Classify image using OpenAI Vision API with semaphore control."""
        async with semaphore:
            try:
                base64_image = self.encode_image_array(image_array)
                
                response = self.client.chat.completions.create(
                    model="gpt-4o",
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {
                                    "type": "text",
                                    "text": "What korean food item is shown in this image? Respond with just the food name (english name if possible) in a single word."
                                },
                                {
                                    "type": "image_url",
                                    "image_url": {
                                        "url": f"data:image/jpeg;base64,{base64_image}"
                                    }
                                }
                            ]
                        }
                    ],
                    max_tokens=50
                )
                
                return response.choices[0].message.content.strip().lower()
                
            except Exception as e:
                print(f"Classification error: {str(e)}")
                return "unknown"

async def process_detections(
    detections: np.ndarray,
    confidences: np.ndarray,
    image_np: np.ndarray,
    sam_predictor,
    classifier: OpenAIClassifier,
    device: str
) -> Tuple[List[Dict], List[np.ndarray]]:
    results = []
    all_masks = []
    
    # Semaphore for limiting to 5 parallel classifications
    semaphore = asyncio.Semaphore(5)
    classification_tasks = []

    for bbox, conf in zip(detections, confidences):
        x1, y1, x2, y2 = map(int, bbox)
        
        # Get crop of the detection
        crop = image_np[y1:y2, x1:x2]
        
        # Add a classification task with semaphore control
        classification_tasks.append(classifier.classify_image(crop, semaphore))
        
        # Generate mask using SAM (synchronously)
        input_box = np.array([x1, y1, x2, y2])
        masks, _, _ = sam_predictor.predict(
            point_coords=None,
            point_labels=None,
            box=input_box[None, :],
            multimask_output=False,
        )
        
        all_masks.append(masks[0])

    # Run classification tasks concurrently with a limit of 5
    class_names = await asyncio.gather(*classification_tasks)
    
    for bbox, conf, class_name in zip(detections, confidences, class_names):
        x1, y1, x2, y2 = map(int, bbox)
        results.append({
            'bbox': [x1, y1, x2, y2],
            'confidence': float(conf),
            'class': class_name
        })
    
    return results, all_masks



In [29]:
image_path = r"C:\Users\user\Documents\GitHub\Korean_Food_Detection\test_exemples\products\Cocacola\46.jpeg"
device =torch.device('cuda' if torch.cuda.is_available() else 'cpu')

output_image = await run_food_detection_and_classification(image_path, PREDICT_ARGS, device)
if output_image:
    output_image.show()
else:
    print("No objects detected in the image.")


0: 640x640 1 food, 64.1ms
Speed: 4.5ms preprocess, 64.1ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)
